# Figure 3: Predictive performance and clinical utility
This notebook reproduces Figure 3A (ROC curves), 3B (calibration curves), 3C (decision curve analysis), and 3D (PPV degradation).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve
from matplotlib import rcParams

rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Helvetica', 'Arial']
rcParams['axes.linewidth'] = 1.5
rcParams['xtick.major.width'] = 1.5
rcParams['ytick.major.width'] = 1.5

color_trauma = '#E64B35'
color_lstm = '#4DBBD5'
color_shock = '#00A087'
color_gray = '#AAAAAA'

data_dir = './data'

In [ ]:
# Load predictions
df = pd.read_excel(f'{data_dir}/Fig3_Performance.xlsx', sheet_name='Fig3_Performance')
y_true = df['y true'].values
y_trauma = df['Trauma-Former'].values
y_lstm = df['LSTM'].values
y_shock = df['Shock Index'].values

# Compute ROC
fpr_trauma, tpr_trauma, _ = roc_curve(y_true, y_trauma)
auc_trauma = auc(fpr_trauma, tpr_trauma)
fpr_lstm, tpr_lstm, _ = roc_curve(y_true, y_lstm)
auc_lstm = auc(fpr_lstm, tpr_lstm)
fpr_shock, tpr_shock, _ = roc_curve(y_true, y_shock)
auc_shock = auc(fpr_shock, tpr_shock)

# Calibration
prob_true_trauma, prob_pred_trauma = calibration_curve(y_true, y_trauma, n_bins=10)
prob_true_lstm, prob_pred_lstm = calibration_curve(y_true, y_lstm, n_bins=10)
prob_true_shock, prob_pred_shock = calibration_curve(y_true, y_shock, n_bins=10)

brier_trauma = brier_score_loss(y_true, y_trauma)
brier_lstm = brier_score_loss(y_true, y_lstm)
brier_shock = brier_score_loss(y_true, y_shock)

In [ ]:
# Decision curve analysis
def dca(y_true, y_score, thresholds):
    n = len(y_true)
    nb = []
    for thresh in thresholds:
        tp = np.sum((y_score >= thresh) & (y_true == 1))
        fp = np.sum((y_score >= thresh) & (y_true == 0))
        nb.append(tp/n - fp/n * (thresh/(1-thresh)))
    return np.array(nb)

thresholds = np.arange(0.01, 0.99, 0.01)
nb_trauma = dca(y_true, y_trauma, thresholds)
nb_lstm = dca(y_true, y_lstm, thresholds)
nb_shock = dca(y_true, y_shock, thresholds)

nb_treat_all = (np.sum(y_true==1)/len(y_true)) - thresholds/(1-thresholds) * (np.sum(y_true==0)/len(y_true))
nb_treat_none = np.zeros_like(thresholds)

# PPV data (from paper)
prevalence_labels = ['Balanced (50%)', 'Realistic (25%)']
ppv_values = [0.89, 0.48]
ppv_ci = [(0.86, 0.92), (0.43, 0.53)]

In [ ]:
# Create figure
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1.2], hspace=0.3, wspace=0.3)

# Panel A: ROC
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(fpr_trauma, tpr_trauma, color=color_trauma, linewidth=2.5,
         label=f'Trauma-Former (AUC={auc_trauma:.3f})')
ax1.plot(fpr_lstm, tpr_lstm, color=color_lstm, linewidth=2,
         label=f'LSTM (AUC={auc_lstm:.3f})')
ax1.plot(fpr_shock, tpr_shock, color=color_shock, linewidth=2,
         label=f'Shock Index (AUC={auc_shock:.3f})')
ax1.plot([0,1], [0,1], 'k--', linewidth=1, alpha=0.5)
ax1.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
ax1.set_ylabel('True Positive Rate (Sensitivity)', fontsize=11)
ax1.set_title('A  ROC curves with MCSE = 0.003', fontsize=12, loc='left', weight='bold')
ax1.legend(loc='lower right', fontsize=9, frameon=False)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.set_xlim([0,1])
ax1.set_ylim([0,1])

# Panel B: Calibration
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot([0,1], [0,1], 'k--', linewidth=1, alpha=0.5, label='Perfect calibration')
ax2.plot(prob_pred_trauma, prob_true_trauma, color=color_trauma, marker='o', linewidth=2, markersize=6,
         label=f'Trauma-Former (Brier={brier_trauma:.2f})')
ax2.plot(prob_pred_lstm, prob_true_lstm, color=color_lstm, marker='s', linewidth=2, markersize=6,
         label=f'LSTM (Brier={brier_lstm:.2f})')
ax2.plot(prob_pred_shock, prob_true_shock, color=color_shock, marker='^', linewidth=2, markersize=6,
         label=f'Shock Index (Brier={brier_shock:.2f})')
ax2.set_xlabel('Mean Predicted Probability', fontsize=11)
ax2.set_ylabel('Observed Proportion', fontsize=11)
ax2.set_title('B  Calibration curves', fontsize=12, loc='left', weight='bold')
ax2.legend(loc='lower right', fontsize=9, frameon=False)
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.set_xlim([0,1])
ax2.set_ylim([0,1])

# Panel C: DCA
ax3 = fig.add_subplot(gs[1, :])
ax3.plot(thresholds, nb_trauma, color=color_trauma, linewidth=2.5, label='Trauma-Former')
ax3.plot(thresholds, nb_lstm, color=color_lstm, linewidth=2, label='LSTM')
ax3.plot(thresholds, nb_shock, color=color_shock, linewidth=2, label='Shock Index')
ax3.plot(thresholds, nb_treat_all, color='gray', linestyle='--', linewidth=2, label='Treat all')
ax3.plot(thresholds, nb_treat_none, color='gray', linestyle=':', linewidth=2, label='Treat none')
ax3.set_xlabel('Threshold Probability', fontsize=11)
ax3.set_ylabel('Net Benefit', fontsize=11)
ax3.set_title('C  Decision curve analysis', fontsize=12, loc='left', weight='bold')
ax3.legend(loc='upper right', fontsize=9, frameon=False)
ax3.grid(True, linestyle='--', alpha=0.5)
ax3.set_xlim([0,1])
ymax = max(np.max(nb_trauma), np.max(nb_treat_all)) * 1.1
ymin = min(np.min(nb_trauma), -0.05)
ax3.set_ylim([ymin, ymax])

# Panel D: PPV bar plot
ax4 = fig.add_subplot(gs[2, :])
bars = ax4.bar(prevalence_labels, ppv_values, color=[color_trauma, color_shock], alpha=0.8,
               edgecolor='black', linewidth=1.5)
yerr_lower = [ppv_values[0]-ppv_ci[0][0], ppv_values[1]-ppv_ci[1][0]]
yerr_upper = [ppv_ci[0][1]-ppv_values[0], ppv_ci[1][1]-ppv_values[1]]
ax4.errorbar(prevalence_labels, ppv_values, yerr=[yerr_lower, yerr_upper],
             fmt='none', ecolor='black', capsize=5, capthick=1.5)
for bar, val in zip(bars, ppv_values):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.2f}', ha='center', va='bottom', fontsize=11, weight='bold')
ax4.annotate('', xy=(1, ppv_values[1]), xytext=(0, ppv_values[0]),
             arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax4.text(0.5, (ppv_values[0]+ppv_values[1])/2 + 0.1, '-46%', ha='center', va='center',
         fontsize=12, color='red', weight='bold')
ax4.text(0.5, (ppv_values[0]+ppv_values[1])/2 - 0.05, 'Alarm Fatigue Zone',
         ha='center', va='center', fontsize=10, color='red', style='italic')
ax4.set_ylabel('Positive Predictive Value', fontsize=11)
ax4.set_title('D  Impact of disease prevalence on PPV', fontsize=12, loc='left', weight='bold')
ax4.set_ylim([0, 1.0])
ax4.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('Figure3.tiff', dpi=300, format='tiff', bbox_inches='tight')
plt.show()